In [1]:
import sys
from pathlib import Path

# Add the src directory to Python path
sys.path.insert(0, str(Path().resolve().parent / 'src'))

from config import EXTERNAL_DATA_DIR, INTERIM_DATA_DIR
import pandas as pd
import requests
from datetime import datetime, timedelta
import time

2026-02-21 18:13:20.238 | INFO     | config:<module>:5 - PROJ_ROOT path is: /Users/vince/Developer/indian_pizza_machine/backend


In [2]:
# Generate all possible weekly date ranges from May 31, 2024 to now
from datetime import datetime, timedelta

# Start date: May 31, 2024 (the earliest known market is may-31 to june-7)
start_date = datetime(2024, 5, 31)
# End date: Today
end_date = datetime.now()

# Generate weekly date ranges (assuming markets run Friday-Friday, 7 days)
weekly_ranges = []
current_date = start_date

while current_date < end_date:
    week_end = current_date + timedelta(days=7)
    weekly_ranges.append({
        'start': current_date,
        'end': week_end
    })
    current_date = week_end

print(f"Generated {len(weekly_ranges)} weekly date ranges")
print(f"From: {weekly_ranges[0]['start'].strftime('%Y-%m-%d')} to {weekly_ranges[0]['end'].strftime('%Y-%m-%d')}")
print(f"To: {weekly_ranges[-1]['start'].strftime('%Y-%m-%d')} to {weekly_ranges[-1]['end'].strftime('%Y-%m-%d')}")

Generated 91 weekly date ranges
From: 2024-05-31 to 2024-06-07
To: 2026-02-20 to 2026-02-27


In [3]:
# Function to generate possible slug patterns for a date range
def generate_slug_patterns(start_date, end_date):
    """Generate possible slug patterns for a given date range."""
    patterns = []
    
    # Both full and abbreviated month names
    start_month_full = start_date.strftime('%B').lower()
    start_month_abbr = start_date.strftime('%b').lower()
    # Polymarket uses "sept" instead of "sep"
    if start_month_abbr == 'sep':
        start_month_abbr = 'sept'
    start_day = start_date.day
    
    # Try end date and end_date ± 1 day (Polymarket sometimes uses different conventions)
    for day_offset in [0, -1, 1]:
        adjusted_end = end_date + timedelta(days=day_offset)
        end_month_full = adjusted_end.strftime('%B').lower()
        end_month_abbr = adjusted_end.strftime('%b').lower()
        # Polymarket uses "sept" instead of "sep"
        if end_month_abbr == 'sep':
            end_month_abbr = 'sept'
        end_day = adjusted_end.day
        
        # Try all combinations of full/abbreviated month names
        for start_m, end_m in [(start_month_full, end_month_full), 
                                (start_month_abbr, end_month_abbr)]:
            # Pattern 1: "elon-musk-of-tweets-{month}-{day}-{month}-{day}"
            patterns.append(f"elon-musk-of-tweets-{start_m}-{start_day}-{end_m}-{end_day}")
            
            # Pattern 2: "of-elon-musk-tweets-between-{month}-{day}-and-{month}-{day}"
            patterns.append(f"of-elon-musk-tweets-between-{start_m}-{start_day}-and-{end_m}-{end_day}")
            
            # Pattern 3: "of-elon-musk-tweets-{month}-{day}-{month}-{day}" (without "between")
            patterns.append(f"of-elon-musk-tweets-{start_m}-{start_day}-{end_m}-{end_day}")
            
            # Pattern 4: Short form when same month (e.g., "oct-11-18" or "october-11-18")
            if start_m == end_m:
                patterns.append(f"elon-musk-of-tweets-{start_m}-{start_day}-{end_day}")
                # Pattern 5: "of-elon-musk-tweets-{month}-{day}-{day}" (short form with "of" prefix)
                patterns.append(f"of-elon-musk-tweets-{start_m}-{start_day}-{end_day}")
    
    # Remove duplicates while preserving order
    seen = set()
    unique_patterns = []
    for p in patterns:
        if p not in seen:
            seen.add(p)
            unique_patterns.append(p)
    
    return unique_patterns

# Function to fetch condition IDs (blobs) from Polymarket API
def get_polymarket_condition_id(slug):
    """
    Fetch the condition ID for a given market slug from Polymarket.
    
    Args:
        slug: The market slug
    
    Returns:
        dict: Market data including condition_id and other metadata, or None if not found
    """
    try:
        url = f"https://gamma-api.polymarket.com/events?slug={slug}"
        response = requests.get(url, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            if data and len(data) > 0:
                event = data[0]
                return {
                    'slug': slug,
                    'condition_id': event.get('conditionId'),
                    'title': event.get('title'),
                    'description': event.get('description'),
                    'end_date': event.get('endDate'),
                    'markets': event.get('markets', [])
                }
        return None
    except Exception as e:
        return None

print("Functions defined successfully")

Functions defined successfully


In [4]:
# Fetch all market blobs by trying different slug patterns
musk_blobs = []
attempted_slugs = []
found_slugs = []
missed_weeks = []  # Track which weeks were not found

print(f"Searching for markets across {len(weekly_ranges)} weekly periods...\n")

for i, date_range in enumerate(weekly_ranges):
    start = date_range['start']
    end = date_range['end']
    
    # Generate possible slug patterns for this date range
    patterns = generate_slug_patterns(start, end)
    
    # Try each pattern
    found = False
    for pattern in patterns:
        attempted_slugs.append(pattern)
        market_data = get_polymarket_condition_id(pattern)
        
        if market_data:
            musk_blobs.append(market_data)
            found_slugs.append(pattern)
            print(f"✓ [{len(musk_blobs)}] {start.strftime('%Y-%m-%d')} to {end.strftime('%Y-%m-%d')}")
            print(f"   {pattern}")
            found = True
            break  # Found it, move to next week
        
        # Small delay between attempts
        time.sleep(0.3)
    
    if not found:
        missed_weeks.append({
            'start': start,
            'end': end,
            'week_num': i + 1
        })
        if i % 10 == 0:
            print(f"  [{i+1}/{len(weekly_ranges)}] Checked {start.strftime('%Y-%m-%d')}... (not found)")

print(f"\n{'='*70}")
print(f"Results:")
print(f"  Attempted: {len(attempted_slugs)} slug patterns")
print(f"  Found: {len(musk_blobs)} markets")
print(f"  Missed: {len(missed_weeks)} weeks")
print(f"  Success rate: {len(musk_blobs)/len(weekly_ranges)*100:.1f}%")
print(f"{'='*70}")

Searching for markets across 91 weekly periods...

✓ [1] 2024-05-31 to 2024-06-07
   of-elon-musk-tweets-between-may-31-and-june-7
✓ [2] 2024-06-07 to 2024-06-14
   of-elon-musk-tweets-june-7-14
✓ [3] 2024-06-14 to 2024-06-21
   elon-musk-of-tweets-june-14-21
✓ [4] 2024-06-21 to 2024-06-28
   elon-musk-of-tweets-june-21-28
✓ [5] 2024-06-28 to 2024-07-05
   elon-musk-of-tweets-june-28-july-5
✓ [6] 2024-07-05 to 2024-07-12
   elon-musk-of-tweets-july-5-july-12
✓ [7] 2024-07-12 to 2024-07-19
   elon-musk-of-tweets-july-12-july-19
✓ [8] 2024-07-19 to 2024-07-26
   elon-musk-of-tweets-july-19-26
✓ [9] 2024-08-02 to 2024-08-09
   elon-musk-of-tweets-august-2-9
✓ [10] 2024-08-09 to 2024-08-16
   elon-musk-of-tweets-august-9-16
✓ [11] 2024-08-16 to 2024-08-23
   elon-musk-of-tweets-august-16-23
✓ [12] 2024-08-23 to 2024-08-30
   elon-musk-of-tweets-august-23-30
✓ [13] 2024-09-06 to 2024-09-13
   elon-musk-of-tweets-sept-6-13
✓ [14] 2024-09-13 to 2024-09-20
   elon-musk-of-tweets-sept-13-20
✓ [

In [5]:
# Display missed weeks
print(f"Missed {len(missed_weeks)} weeks:\n")

for missed in missed_weeks:
    start = missed['start']
    end = missed['end']
    week_num = missed['week_num']
    
    print(f"Week #{week_num}: {start.strftime('%Y-%m-%d')} to {end.strftime('%Y-%m-%d')}")
    print(f"  ({start.strftime('%B %d')} to {end.strftime('%B %d, %Y')})")
    
    # Show what patterns were tried for this week
    patterns = generate_slug_patterns(start, end)
    print(f"  Tried {len(patterns)} patterns:")
    for i, p in enumerate(patterns[:3], 1):  # Show first 3
        print(f"    {i}. {p}")
    if len(patterns) > 3:
        print(f"    ... and {len(patterns) - 3} more")
    print()

# Create a DataFrame of missed weeks
if len(missed_weeks) > 0:
    missed_df = pd.DataFrame([
        {
            'week_num': m['week_num'],
            'start_date': m['start'].strftime('%Y-%m-%d'),
            'end_date': m['end'].strftime('%Y-%m-%d'),
            'date_range': f"{m['start'].strftime('%b %d')} - {m['end'].strftime('%b %d, %Y')}"
        }
        for m in missed_weeks
    ])
    print("\nMissed Weeks DataFrame:")
    display(missed_df)

Missed 3 weeks:

Week #9: 2024-07-26 to 2024-08-02
  (July 26 to August 02, 2024)
  Tried 18 patterns:
    1. elon-musk-of-tweets-july-26-august-2
    2. of-elon-musk-tweets-between-july-26-and-august-2
    3. of-elon-musk-tweets-july-26-august-2
    ... and 15 more

Week #14: 2024-08-30 to 2024-09-06
  (August 30 to September 06, 2024)
  Tried 18 patterns:
    1. elon-musk-of-tweets-august-30-september-6
    2. of-elon-musk-tweets-between-august-30-and-september-6
    3. of-elon-musk-tweets-august-30-september-6
    ... and 15 more

Week #46: 2025-04-11 to 2025-04-18
  (April 11 to April 18, 2025)
  Tried 30 patterns:
    1. elon-musk-of-tweets-april-11-april-18
    2. of-elon-musk-tweets-between-april-11-and-april-18
    3. of-elon-musk-tweets-april-11-april-18
    ... and 27 more


Missed Weeks DataFrame:


,week_num,start_date,end_date,date_range
0,9,2024-07-26,2024-08-02,"Jul 26 - Aug 02, 2024"
1,14,2024-08-30,2024-09-06,"Aug 30 - Sep 06, 2024"
2,46,2025-04-11,2025-04-18,"Apr 11 - Apr 18, 2025"


In [6]:
# Display the collected market blobs
print("Musk Tweet Markets:\n")

for i, blob in enumerate(musk_blobs, 1):
    print(f"{i}. {blob['title']}")
    print(f"   Slug: {blob['slug']}")
    print(f"   Condition ID: {blob['condition_id']}")
    print(f"   End Date: {blob['end_date']}")
    print(f"   Markets: {len(blob.get('markets', []))}")
    print()

# Create a DataFrame for easy analysis
musk_markets_df = pd.DataFrame([
    {
        'slug': blob['slug'],
        'condition_id': blob['condition_id'],
        'title': blob['title'],
        'end_date': blob['end_date'],
        'num_markets': len(blob.get('markets', []))
    }
    for blob in musk_blobs
])

print("\nMarkets DataFrame:")
musk_markets_df

Musk Tweet Markets:

1. Elon Musk # of tweets May 31 - June 7?
   Slug: of-elon-musk-tweets-between-may-31-and-june-7
   Condition ID: None
   End Date: 2024-06-07T12:00:00Z
   Markets: 8

2. Elon Musk # of tweets June 7-14?
   Slug: of-elon-musk-tweets-june-7-14
   Condition ID: None
   End Date: 2024-06-14T12:00:00Z
   Markets: 8

3. Elon Musk # of tweets June 14-21?
   Slug: elon-musk-of-tweets-june-14-21
   Condition ID: None
   End Date: 2024-06-21T12:00:00Z
   Markets: 8

4. Elon Musk # of tweets June 21-28?
   Slug: elon-musk-of-tweets-june-21-28
   Condition ID: None
   End Date: 2024-06-28T12:00:00Z
   Markets: 8

5. Elon Musk # of tweets June 28 - July 5?
   Slug: elon-musk-of-tweets-june-28-july-5
   Condition ID: None
   End Date: 2024-07-05T12:00:00Z
   Markets: 8

6. Elon Musk # of tweets July 5 - July 12?
   Slug: elon-musk-of-tweets-july-5-july-12
   Condition ID: None
   End Date: 2024-07-12T12:00:00Z
   Markets: 8

7. Elon Musk # of tweets July 12 - July 19?
   Slug: 

,slug,condition_id,title,end_date,num_markets
0,of-elon-musk-tweets-between-may-31-and-june-7,None,Elon Musk # of tweets May 31 - June 7?,2024-06-07T12:00:00Z,8
1,of-elon-musk-tweets-june-7-14,None,Elon Musk # of tweets June 7-14?,2024-06-14T12:00:00Z,8
2,elon-musk-of-tweets-june-14-21,None,Elon Musk # of tweets June 14-21?,2024-06-21T12:00:00Z,8
3,elon-musk-of-tweets-june-21-28,None,Elon Musk # of tweets June 21-28?,2024-06-28T12:00:00Z,8
4,elon-musk-of-tweets-june-28-july-5,None,Elon Musk # of tweets June 28 - July 5?,2024-07-05T12:00:00Z,8
...,...,...,...,...,...
83,elon-musk-of-tweets-january-23-january-30,None,"Elon Musk # tweets January 23 - January 30, 2026?",2026-01-30T17:00:00Z,38
84,elon-musk-of-tweets-january-30-february-6,None,"Elon Musk # tweets January 30 - February 6, 2026?",2026-02-06T17:00:00Z,38
85,elon-musk-of-tweets-february-6-february-13,None,"Elon Musk # tweets February 6 - February 13, 2...",2026-02-13T17:00:00Z,30
86,elon-musk-of-tweets-february-13-february-20,None,"Elon Musk # tweets February 13 - February 20, ...",2026-02-20T17:00:00Z,30


In [7]:
# Save musk_blobs to a JSON file for later use
import json

output_file = INTERIM_DATA_DIR / 'musk_polymarket_blobs.json'

with open(output_file, 'w') as f:
    json.dump(musk_blobs, f, indent=2)

print(f"Saved {len(musk_blobs)} market blobs to {output_file}")

# Also save the DataFrame as parquet
df_output = INTERIM_DATA_DIR / 'musk_polymarket_markets.parquet'
musk_markets_df.to_parquet(df_output, index=False)
print(f"Saved markets DataFrame to {df_output}")

Saved 88 market blobs to /Users/vince/Developer/indian_pizza_machine/backend/data/interim/musk_polymarket_blobs.json
Saved markets DataFrame to /Users/vince/Developer/indian_pizza_machine/backend/data/interim/musk_polymarket_markets.parquet


In [8]:
import json

with open(output_file, 'r') as f:
    loaded_blobs = json.load(f)

print(f"Loaded {len(loaded_blobs)} blobs from {output_file}")

Loaded 88 blobs from /Users/vince/Developer/indian_pizza_machine/backend/data/interim/musk_polymarket_blobs.json


In [9]:
# Fetch and add the missing market
missing_slug = "elon-musk-of-tweets-april-4-11-lower-brackets"

print(f"Fetching missing market: {missing_slug}")
missing_market = get_polymarket_condition_id(missing_slug)

if missing_market:
    # Load current JSON
    with open(output_file, 'r') as f:
        current_blobs = json.load(f)
    
    # Check if already exists
    existing_slugs = [blob['slug'] for blob in current_blobs]
    
    if missing_slug not in existing_slugs:
        # Add the missing market
        current_blobs.append(missing_market)
        
        # Save updated JSON
        with open(output_file, 'w') as f:
            json.dump(current_blobs, f, indent=2)
        
        print(f"✓ Added missing market!")
        print(f"  Title: {missing_market['title']}")
        print(f"  Condition ID: {missing_market['condition_id']}")
        print(f"  Total markets now: {len(current_blobs)}")
        
        # Update the DataFrame too
        musk_blobs = current_blobs
        musk_markets_df = pd.DataFrame([
            {
                'slug': blob['slug'],
                'condition_id': blob['condition_id'],
                'title': blob['title'],
                'end_date': blob['end_date'],
                'num_markets': len(blob.get('markets', []))
            }
            for blob in musk_blobs
        ])
        df_output = INTERIM_DATA_DIR / 'musk_polymarket_markets.parquet'
        musk_markets_df.to_parquet(df_output, index=False)
        print(f"✓ Updated parquet file")
    else:
        print(f"Market already exists in JSON")
else:
    print(f"✗ Failed to fetch market")

Fetching missing market: elon-musk-of-tweets-april-4-11-lower-brackets
✓ Added missing market!
  Title: Elon Musk # of tweets April 4-11? (Lower Brackets)
  Condition ID: None
  Total markets now: 89
✓ Updated parquet file
